# Demo: run the plan-review agent on the test bench

The agent reviews a BLM mine Plan of Operations (Thacker Pass) and flags what's missing, vague, inconsistent or implausible.

The bench has 41 copies of the plan:
- **TP-000** is the clean plan;
- **TP-001 to TP-040** each have one planted flaw.

A flag counts as a **catch** if it names the right checklist item at the right place in the plan.

**Before you start:** put `OPENROUTER_API_KEY` in `.env`, and open this notebook with the project's `.venv` kernel.

Answers already in DSPy's cache are free. New ones cost money, and the project budget in `.env` stops the run if it runs out.

In [ ]:
# Setup: work from the project root so the project's packages import
import json, os, subprocess, sys
from pathlib import Path

os.chdir(Path.cwd() if (Path.cwd() / "harness").exists() else Path.cwd().parent)
sys.path.insert(0, os.getcwd())

from harness.run import run

In [ ]:
# Settings
MODEL = "dev"      # "dev" = gpt-6-luna (cheap), "strong" = gpt-6-sol
SYSTEM = "v2"      # "v2" = checklist + value checks, "v1" = checklist only
RUN_NAME = f"demo-{SYSTEM}-{MODEL}"

In [ ]:
# Run the agent on all 41 cases, then score the reports
summary = run(SYSTEM, cases="all", name=RUN_NAME, model=MODEL, quiet=True)
subprocess.run([sys.executable, "-m", "bench.score", "thacker_pass_clean", summary["runs_dir"]],
               check=True, capture_output=True)

print(f"{summary['n_cases']} cases, {summary['lm_calls']} model calls, "
      f"{summary['seconds'] / 60:.1f} min, ${summary['cost_usd'] or 0:.2f}")

In [ ]:
# Results
scores = json.loads((Path(summary["runs_dir"]) / "scores.json").read_text())
s = scores["summary"]

print(f"Recall:    {s['recall']:.2f}  (planted flaws caught)")
print(f"Precision: {s['precision']:.3f}")
print(f"False alarms: {s['false_alarms_total']} in total, {s['false_alarms_on_controls']} on the clean plan")

print("\nRecall by type:")
for name, g in s["by_type"].items():
    print(f"  {name:13} {g['recall']:.2f}  ({g['n']} cases)")

print("\nRecall by difficulty:")
for name, g in s["by_difficulty"].items():
    print(f"  {name:13} {g['recall']:.2f}  ({g['n']} cases)")

In [ ]:
# Each flawed case: caught or not
for row in scores["cases"]:
    if row["planted"]:
        mark = "caught" if row["caught"] else "-"
        print(f"{row['case_id']}  {row['type']:13} {row['difficulty']:7} {mark}")